# Toy DGD: Two Uniform Blobs in 10D -> 2D Latent, 2-Component GMM

The smallest possible instance of the mechanism used everywhere else in this repo (see `dgd_training_demo.ipynb`): a decoder and per-sample latents optimized directly (no encoder), regularized by a Gaussian-mixture prior fit with [`tgmm`](https://adriansousapoza.github.io/tgmm/). Data here is synthetic and low-dimensional enough that training takes well under a second and the latent space is directly plottable -- no PCA/UMAP needed for the 2D latent (only for glancing at the raw 10D data). Runs on CPU, no RAPIDS/GPU required.

## The math

**Data** ($i = 1, \dots, N$, two uniform blobs in $\mathbb{R}^{10}$, label $y_i$ never seen by the model):

$$
x_i = c_{y_i} + u_i, \qquad u_i \sim \mathrm{Unif}([-r, r]^{10}), \qquad y_i \in \{1, 2\}
$$

**Model** -- decoder $f_\theta: \mathbb{R}^2 \to \mathbb{R}^{10}$ (a small MLP) and free per-sample latents $z_i \in \mathbb{R}^2$, regularized by a $K{=}2$-component Gaussian-mixture prior:

$$
\mathcal{L}(\theta, Z) = \sum_{i=1}^N \|f_\theta(z_i) - x_i\|_2^2 \;-\; \lambda \sum_{i=1}^N \log p_{\text{GMM}}(z_i), \qquad p_{\text{GMM}}(z) = \sum_{c=1}^{2} \pi_c\, \mathcal{N}(z; \mu_c, \Sigma_c)
$$

**Optimization** -- block-coordinate: gradient steps on $\theta$ and $Z$ every epoch (one AdamW optimizer over both here, for simplicity -- the main pipeline uses separate optimizers for the decoder and the representations since they want different learning rates), with a reconstruction-only warm-up before the GMM term is added, and the GMM periodically refit via EM to the current $Z$. No noise injection here (see `noise_injection_explained.ipynb` for why the main pipeline uses it) -- with only two well-separated clusters at this scale, plain MAP already gives a clean, non-degenerate latent space.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

current_dir = Path.cwd()
project_root = current_dir.parent if 'notebooks' in current_dir.parts else current_dir
sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.models import RepresentationLayer
from tgmm import GaussianMixture, ClusteringMetrics
from tgmm.plotting import plot_gmm

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')  # tiny problem, no need for a GPU

In [ ]:
N_per_blob = 150
dim_x = 10
r = 1.0

c1 = -3.0 * torch.ones(dim_x)
c2 = 3.0 * torch.ones(dim_x)

x1 = c1 + (torch.rand(N_per_blob, dim_x) * 2 - 1) * r
x2 = c2 + (torch.rand(N_per_blob, dim_x) * 2 - 1) * r

x = torch.cat([x1, x2], dim=0)
y_true = torch.cat([torch.zeros(N_per_blob, dtype=torch.long), torch.ones(N_per_blob, dtype=torch.long)])
N = x.shape[0]

print(f"x: {tuple(x.shape)} -- two blobs of {N_per_blob} points each in {dim_x}D, "
      f"half-width r={r}, centers at {c1[0].item():.0f}*1 and {c2[0].item():.0f}*1")

In [ ]:
pca_raw = PCA(n_components=2, random_state=42)
x_pca = pca_raw.fit_transform(x.numpy())

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x_pca[:, 0], x_pca[:, 1], c=y_true.numpy(), cmap='coolwarm', s=15, alpha=0.7)
ax.set_title(f"Raw 10D data, PCA projection\n({pca_raw.explained_variance_ratio_.sum()*100:.1f}% variance explained)")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()

## Model and training

A 2-hidden-layer MLP decoder and a `RepresentationLayer` for the free latents, exactly the same class the main pipeline uses (`dist='uniform'` here instead of the default `'zeros'`, since there's no noise injection to break the initial symmetry).

In [ ]:
dim_z = 2

decoder = nn.Sequential(
    nn.Linear(dim_z, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, dim_x),
)

rep = RepresentationLayer(
    dim=dim_z,
    n_samples=N,
    dist='uniform',
    dist_params={'low': -0.5, 'high': 0.5},
    device=device,
)

optimizer = torch.optim.AdamW(list(decoder.parameters()) + list(rep.parameters()), lr=0.02)
print(f"Decoder: {sum(p.numel() for p in decoder.parameters())} params. Representations: {rep.n_rep} x {rep.dim}")

In [ ]:
epochs = 300
first_epoch_gmm = 60   # reconstruction-only warm-up before the GMM term is added
refit_every = 20       # epochs between GMM refits, once active
lambda_gmm = 1.0

gmm = None
cluster_metrics = ClusteringMetrics()
history = {'loss': [], 'recon': [], 'gmm': []}

for epoch in range(1, epochs + 1):
    optimizer.zero_grad()
    z = rep()  # full batch -- no indices needed, ixs=None returns every representation
    x_hat = decoder(z)
    recon_loss = F.mse_loss(x_hat, x, reduction='sum')

    if epoch >= first_epoch_gmm:
        if gmm is None:
            gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
            with torch.no_grad():
                gmm.fit(z.detach())
        elif epoch % refit_every == 0:
            with torch.no_grad():
                gmm.fit(z.detach(), warm_start=True)
        gmm_loss = -lambda_gmm * gmm.score_samples(z).sum()
        loss = recon_loss + gmm_loss
    else:
        gmm_loss = torch.tensor(0.0)
        loss = recon_loss

    loss.backward()
    optimizer.step()

    history['loss'].append(loss.item())
    history['recon'].append(recon_loss.item())
    history['gmm'].append(gmm_loss.item())

    if epoch % 50 == 0 or epoch == epochs:
        print(f"epoch {epoch:3d} | loss {loss.item():9.3f} | recon {recon_loss.item():9.3f} | gmm {gmm_loss.item():9.3f}")

print("Training complete.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history['loss'], label='total loss')
ax.plot(history['recon'], label='reconstruction', alpha=0.7)
ax.axvline(first_epoch_gmm, color='gray', linestyle='--', alpha=0.5, label='GMM term added')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.set_title('Training curve')
plt.tight_layout()
plt.show()

## The learned latent space

$z$ is already 2D, so this *is* the latent space -- no PCA/UMAP projection needed (unlike the raw 10D data above, or the 8D+ latents in the main pipeline). Each cluster below tends to collapse into a thin sliver rather than filling out a round blob -- with no noise injection pulling representations apart, and no reconstructible signal beyond "which cluster" (more on this in Reconstruction quality below), $z$ has no incentive to use its full 2 dimensions per cluster.

In [ ]:
z_final = rep().detach()

fig, ax = plt.subplots(figsize=(6, 6))
plot_gmm(
    z_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_true, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Learned 2D latent space + GMM components",
    xlabel="z[0]", ylabel="z[1]",
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
z_pred = gmm.predict(z_final)
ami = cluster_metrics.adjusted_mutual_info_score(y_true, z_pred)
ari = cluster_metrics.adjusted_rand_score(y_true, z_pred)
print(f"GMM clusters vs. true blob labels: AMI={ami:.4f}, ARI={ari:.4f} (1.0 = perfect recovery)")

## Reconstruction quality

A 2D $z$ has room to encode *which blob* a point came from, plus 2 real numbers of position within it -- the rest of each point's position across all 10 dimensions is independent uniform noise, which is information-theoretically impossible to recover from any summary, however good. So the fair comparison isn't "does $f_\theta(z_i)$ reconstruct $x_i$ exactly" (it can't), it's: how close does it get to an *oracle* that's told the true blob and just predicts that blob's center?

In [ ]:
with torch.no_grad():
    x_hat_final = decoder(z_final)

x_hat_pca = pca_raw.transform(x_hat_final.numpy())  # same fitted PCA as the raw-data plot above

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, data, title in [(axes[0], x_pca, "Original x"), (axes[1], x_hat_pca, "Reconstructed decoder(z)")]:
    ax.scatter(data[:, 0], data[:, 1], c=y_true.numpy(), cmap='coolwarm', s=15, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("PC 1 (of original x)")
    ax.set_ylabel("PC 2 (of original x)")
plt.tight_layout()
plt.show()

mse_model = F.mse_loss(x_hat_final, x).item()
mse_no_info = F.mse_loss(x.mean(dim=0, keepdim=True).expand_as(x), x).item()
cluster_centers = torch.stack([c1, c2])[y_true]
mse_oracle = F.mse_loss(cluster_centers, x).item()

print(f"MSE, no information (predict global mean):        {mse_no_info:.4f}")
print(f"MSE, oracle (told the true blob, predicts center): {mse_oracle:.4f}")
print(f"MSE, model reconstruction decoder(z):              {mse_model:.4f}")

## Takeaway

Same three ingredients as `dgd_training_demo.ipynb` -- a decoder, free per-sample latents (no encoder), and a GMM prior fit with EM -- just small enough to watch converge directly. Scaling this up to images means: a convolutional decoder instead of an MLP, many more latent dimensions (so the latent space itself needs PCA/UMAP to look at, same as the raw 10D data here), more GMM components, and noise injection to keep the decoder well-behaved between training points (see `noise_injection_explained.ipynb`) -- but the objective being optimized is exactly this one.